# RNN from scratch : construire une architecture récurrente pas à pas

Vincent Guigue (vincent.guigue@agroparistech.fr)

Notebook d'échauffement, peu utile si vous avez suivi le cours en détail : c'est une marche intermédiaire avant le premier TP sur les séries temporelles (4_1).

**Objectifs**
- comprendre ce que calcule une cellule récurrente et comment on « déroule » la récurrence sur une séquence ;
- maîtriser les **dimensions** des tenseurs manipulés (temps × batch × features) ;
- passer d'une implémentation à la main au module `nn.RNN` de PyTorch (puis LSTM / GRU).

**Convention de ce module** : comme dans PyTorch par défaut, les séquences sont stockées *temps en premier* : `seq_x` est de taille `T x B x d` (longueur, batch, dimension). L'option `batch_first=True` des modules récurrents permet l'autre convention.

## Table des matières

- [0. Mise en place : librairies, device, TensorBoard](#0-mise-en-place-librairies-device-tensorboard)
- [A. Première idée : construire la cellule récurrente](#a-premiere-idee-construire-la-cellule-recurrente)
    - [A.1 Création du réseau](#a-1-creation-du-reseau)
    - [A.2 Faire passer les données dans le réseau](#a-2-faire-passer-les-donnees-dans-le-reseau)
    - [A.3 RNN et récurrence](#a-3-rnn-et-recurrence)
- [B. RNN V2 : la récurrence dans le `forward`](#b-rnn-v2-la-recurrence-dans-le-forward)
- [C. RNN V3 : utiliser le module `nn.RNN`](#c-rnn-v3-utiliser-le-module-nn-rnn)
- [D. Pour aller plus loin](#d-pour-aller-plus-loin)

<a id="0-mise-en-place-librairies-device-tensorboard"></a>
## 0. Mise en place : librairies, device, TensorBoard

Aucun apprentissage dans ce notebook : TensorBoard n'est pas utilisé, mais la cellule est conservée pour homogénéité avec les autres TP.

In [1]:
import torch
from torch import nn
from torch.nn import functional as F

In [2]:
# ---- Choix du device ----
# Pour ces petits RNN, la récurrence est une boucle séquentielle sur le temps : sur Mac, le GPU (MPS)
# est en général PLUS LENT que le CPU (mesuré sur le TP métro : 12 s/epoch en MPS contre 1,2 s/epoch en CPU).
# MPS n'est donc activé que sur demande. Avec un GPU NVIDIA (CUDA), le gain reste réel.
USE_MPS = False

if torch.cuda.is_available():
    device = torch.device("cuda")
elif USE_MPS and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("torch :", torch.__version__, "| device :", device)
_ = torch.manual_seed(0)  # reproductibilité

torch : 2.11.0+cu128 | device : cuda


In [5]:
# ---- TensorBoard ----
from pathlib import Path
TB_PATH = str(Path("./logs").absolute())    # une expérience = un sous-répertoire : SummaryWriter(TB_PATH / nom)
try:
    from torch.utils.tensorboard import SummaryWriter
except ImportError:                    # tensorboard absent : on continue sans enregistrer les courbes
    class SummaryWriter:
        def __init__(self, *args, **kwargs): pass
        def add_scalar(self, *args, **kwargs): pass
        def close(self): pass
    print("tensorboard non installé : les courbes ne seront pas enregistrées")

# Option 1 (recommandée) : dans un terminal, lancer la commande ci-dessous puis ouvrir http://localhost:6006
print(f"tensorboard --logdir {TB_PATH}")

# Option 2 : TensorBoard directement dans le notebook (décommenter les 2 lignes)
# %load_ext tensorboard
# %tensorboard --logdir {TB_PATH}

tensorboard --logdir c:\cours_code\Python\DeepLearning_vguigue\4-RNN\notebooks\logs


<a id="a-premiere-idee-construire-la-cellule-recurrente"></a>
## A. Première idée : construire la cellule récurrente

C'est l'approche des tutoriels officiels (remarquables) de PyTorch :
- [NLP From Scratch: Classifying Names with a Character-Level RNN](https://pytorch.org/tutorials/intermediate/char_rnn_classification_tutorial.html)
- [NLP From Scratch: Generating Names with a Character-Level RNN](https://pytorch.org/tutorials/intermediate/char_rnn_generation_tutorial.html)

On rappelle la structure classique en classification de signaux (*many-to-one*) : la séquence est lue entièrement, seule la dernière couche cachée sert à la prédiction.

![Image RNN](data/rnn_unfold2.png)

$$h_t = \tanh(W_1 x_t + W_2 h_{t-1}), \qquad \hat y = W_3 h_T$$

<a id="a-1-creation-du-reseau"></a>
### A.1 Création du réseau

1. Réseau correspondant au schéma ci-dessus (une seule étape de récurrence)
2. Données arbitraires
3. Instanciation du réseau + utilisation sur un exemple
4. Passage dans une loss pour vérifier la cohérence des dimensions

**Remarque** : le tutoriel PyTorch termine le réseau par un `LogSoftmax` (à combiner avec `NLLLoss`). Nous renvoyons directement les **logits** et utiliserons `CrossEntropyLoss` (qui intègre le softmax) : c'est équivalent, plus simple et numériquement plus stable.

In [6]:
class RNN_V1(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.i2h = nn.Linear(input_size, hidden_size)   # W1
        self.h2h = nn.Linear(hidden_size, hidden_size)  # W2
        self.h2o = nn.Linear(hidden_size, output_size)  # W3

    def forward(self, input, hidden=None):
        if hidden is None:                              # init par défaut à 0
            hidden = self.initHidden().to(input.device)
        hidden = torch.tanh(self.i2h(input) + self.h2h(hidden))
        output = self.h2o(hidden)                       # logits
        return output, hidden

    def initHidden(self):
        return torch.zeros(1, self.hidden_size)  # 1 x h : diffusé (broadcast) sur tout le batch

In [7]:
# 2. Données arbitraires (x_1,...,x_T) :
#   dim0 : longueur de la séquence T = 23
#   dim1 : batch de taille 10
#   dim2 : x_t dans R^100
seq_x = torch.rand(23, 10, 100)
y     = torch.randint(0, 2, (10,))  # une étiquette (2 classes) par séquence du batch
print(y)

tensor([0, 1, 0, 0, 1, 1, 1, 1, 0, 0])


In [8]:
# 3. Instanciation
n_hidden = 50
d_data   = 100
n_cl     = 2
rnn = RNN_V1(d_data, n_hidden, n_cl)

<a id="a-2-faire-passer-les-donnees-dans-le-reseau"></a>
### A.2 Faire passer les données dans le réseau

Qu'a-t-on le droit de donner à ce réseau ? Avez-vous remarqué qu'il n'est **pas récurrent** ?

Exécuter les deux propositions suivantes, comparer les tailles obtenues et réfléchir aux conséquences : que calcule réellement chacune d'elles ?

In [9]:
# PROPOSITION 1 : toute la séquence d'un coup
yhat, hidden = rnn(seq_x)
print("yhat:  ", yhat.size())
print("hidden:", hidden.size())

yhat:   torch.Size([23, 10, 2])
hidden: torch.Size([23, 10, 50])


In [10]:
# PROPOSITION 2 : uniquement le premier pas de temps
yhat, hidden = rnn(seq_x[0])
print("yhat:  ", yhat.size())
print("hidden:", hidden.size())

yhat:   torch.Size([10, 2])
hidden: torch.Size([10, 50])


In [ ]:
# Réponse :
##  TODO 
# Réponse :
# PROPOSITION 1 : rnn(seq_x) avec seq_x de taille T x B x d = 23 x 10 x 100.
#   nn.Linear agit sur la DERNIÈRE dimension et diffuse sur les autres, donc ça "marche" :
#   yhat : 23 x 10 x 2, hidden : 23 x 10 x 50.
#   Mais ce n'est pas une récurrence. Chaque pas de temps est traité indépendamment,
#   avec h_{t-1} = 0 (hidden=None), puis h2h(0) = biais seulement.
#   Le réseau est un simple MLP appliqué en parallèle sur chaque x_t.
#
# PROPOSITION 2 : rnn(seq_x[0]) avec seq_x[0] de taille B x d = 10 x 100.
#   yhat : 10 x 2, hidden : 10 x 50.
#   Les dimensions sont correctes, mais on n'utilise que x_1 et on ignore
#   les 22 autres pas de temps : aucune information de la séquence n'est exploitée.
#
# Conclusion : le réseau ne fait qu'UNE étape de récurrence. Il faut écrire
# la boucle sur t à l'extérieur en propageant hidden (A.3), puis à l'intérieur du forward (B).

<a id="a-3-rnn-et-recurrence"></a>
### A.3 RNN et récurrence

Traiter une séquence entière sans propager les états cachés n'a aucun intérêt... Mais ne traiter que le premier pas de temps non plus.

Écrire la boucle de récurrence à l'extérieur du réseau, puis vérifier les dimensions et calculer la loss (étape 4).

In [14]:
# Passage de l'ensemble de la séquence : la boucle est à l'extérieur du réseau
##  TODO 
hidden = None
for t in range(seq_x.size(0)):
    yhat, hidden = rnn(seq_x[t], hidden)

print("yhat:  ", yhat.size())
print("hidden:", hidden.size())

# 4. Vérification de la cohérence avec la loss
loss = nn.CrossEntropyLoss()
print("loss:", loss(yhat, y).item())

yhat:   torch.Size([10, 2])
hidden: torch.Size([10, 50])
loss: 0.7263590693473816


<a id="b-rnn-v2-la-recurrence-dans-le-forward"></a>
## B. RNN V2 : la récurrence dans le `forward`

La solution actuelle est insatisfaisante : le `forward` est non standard, on ne peut pas intégrer cette architecture dans notre boucle d'apprentissage classique (`yhat = model(x)`).

Intégrer la boucle de récurrence à l'intérieur du `forward` pour avoir :
- une séquence en entrée (`T x B x d`) ;
- en sortie : la prédiction (calculée sur le dernier état caché) **et** la séquence de tous les états cachés (`T x B x h`).

Indice : `torch.stack(liste, dim=0)` empile une liste de tenseurs de même taille le long d'une nouvelle dimension.

In [ ]:
class RNN_V2(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.i2h = nn.Linear(input_size, hidden_size)   # W1
        self.h2h = nn.Linear(hidden_size, hidden_size)  # W2
        self.h2o = nn.Linear(hidden_size, output_size)  # W3

    def forward(self, input, hidden=None):
        if hidden is None:
            hidden = self.initHidden().to(input.device)
        allh = []
        # 1. calculer tous les états cachés et les ajouter dans la liste allh
        # 2. former un tenseur T x B x h avec tous les éléments de la liste
        ##  TODO 
        output = self.h2o(hidden)   # prédiction à partir du dernier état caché
        return output, allh

    def initHidden(self):
        return torch.zeros(1, self.hidden_size)  # 1 x h : diffusé (broadcast) sur tout le batch

Vérifier les dimensions attendues dans la boîte ci-dessous : `yhat` doit être `B x n_cl` et `allh` doit être `T x B x n_hidden`.

In [ ]:
rnn = RNN_V2(d_data, n_hidden, n_cl)
yhat, allh = rnn(seq_x)
print("yhat:", yhat.size())
print("allh:", allh.size())
print("loss:", loss(yhat, y).item())

<a id="c-rnn-v3-utiliser-le-module-nn-rnn"></a>
## C. RNN V3 : utiliser le module `nn.RNN`

On utilise maintenant un module prêt à l'emploi ([doc `nn.RNN`](https://pytorch.org/docs/stable/generated/torch.nn.RNN.html)) ; on basculera ensuite facilement vers des LSTM ou des GRU.

**Rappel de cours** : `seq_h, last_h = nn.RNN(...)(seq_x)` renvoie
- `seq_h` : tous les états cachés de la **dernière couche**, taille `T x B x h` ;
- `last_h` : le dernier état caché de **chaque couche**, taille `num_layers x B x h` (attention à cette première dimension !).

Exécuter le rappel ci-dessous, puis utiliser ce module à l'intérieur de votre proposition.

In [ ]:
# Rappel de cours
model = nn.RNN(input_size=100, hidden_size=50, num_layers=1, nonlinearity='tanh', bias=True)

seq_x = torch.rand(23, 10, 100)  # T x B x d
seq_h, last_h = model(seq_x)     # pour de la classification, on utilise last_h

print("seq_h : ", seq_h.size())
print("last_h: ", last_h.size())

In [ ]:
class RNN_V3(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.rec = nn.RNN(input_size=input_size, hidden_size=hidden_size,
                          num_layers=1, nonlinearity='tanh')  # activation par défaut... on peut en tester d'autres
        self.h2o = nn.Linear(hidden_size, output_size)        # W3

    def forward(self, input, hidden=None):
        # 1. calculer tous les états cachés d'un coup (nn.RNN accepte hidden=None => init à 0)
        # 2. calculer la sortie à partir du dernier état caché : attention à la taille de last_h !
        ##  TODO 
        return output, allh

Vérifier les dimensions attendues dans la boîte ci-dessous (mêmes tailles que pour `RNN_V2`).

In [ ]:
rnn = RNN_V3(d_data, n_hidden, n_cl)
yhat, allh = rnn(seq_x)
print("yhat:", yhat.size())
print("allh:", allh.size())
print("loss:", loss(yhat, y).item())

<a id="d-pour-aller-plus-loin"></a>
## D. Pour aller plus loin

1. **LSTM / GRU** : modifier `RNN_V3` pour pouvoir choisir le module récurrent. Attention, le `forward` d'un `nn.LSTM` renvoie `(seq_h, (h_n, c_n))` : l'état caché est un *couple* (état caché, cellule mémoire).
2. **Many-to-many** : on veut maintenant une prédiction à chaque pas de temps (sortie `T x B x n_cl`). Que faut-il changer dans le `forward` ? Comment utiliser la `CrossEntropyLoss` qui attend une entrée `N x C` (ou `N x C x d`) ?

In [ ]:
# 1. LSTM / GRU
##  TODO 

In [ ]:
# 2. Many-to-many : une étiquette par pas de temps
y_seq = torch.randint(0, n_cl, (23, 10))   # T x B
##  TODO 